In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

plt.style.use('dark_background')
sns.set_palette('husl')

## 1. Загрузка данных

In [ ]:
train_x = np.load('../Data/train_x.npy', allow_pickle=True)
train_y = np.load('../Data/train_y.npy', allow_pickle=True)
valid_x = np.load('../Data/valid_x.npy', allow_pickle=True)
valid_y = np.load('../Data/valid_y.npy', allow_pickle=True)

print(f'train_x: {train_x.shape}, dtype={train_x.dtype}')
print(f'train_y: {train_y.shape}, dtype={train_y.dtype}')
print(f'valid_x: {valid_x.shape}, dtype={valid_x.dtype}')
print(f'valid_y: {valid_y.shape}, dtype={valid_y.dtype}')
print(f'\nПример train_y[0]: {train_y[0]}')
print(f'Пример valid_y[0]: {valid_y[0]}')

## 2. Анализ меток — восстановление классов

In [ ]:
from ml.preprocess import extract_label, build_label_map, encode_labels

train_labels = [extract_label(l) for l in train_y]
valid_labels = [extract_label(l) for l in valid_y]

label_map = build_label_map(np.concatenate([train_y, valid_y]))

print('Классы (20 планет):')
for name, idx in label_map.items():
    print(f'  {idx:2d}: {name}')

In [ ]:
train_counts = Counter(train_labels)
valid_counts = Counter(valid_labels)

df_counts = pd.DataFrame({
    'class': list(train_counts.keys()),
    'train': list(train_counts.values()),
    'valid': [valid_counts.get(k, 0) for k in train_counts.keys()]
}).sort_values('train', ascending=False).reset_index(drop=True)

print(df_counts.to_string())
print(f'\nИтого: train={df_counts.train.sum()}, valid={df_counts.valid.sum()}')

## 3. Распределение классов

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

df_sorted = df_counts.sort_values('train', ascending=True)

axes[0].barh(df_sorted['class'], df_sorted['train'], color='#00b4d8')
axes[0].set_title('Train — записей по классу', fontsize=14)
axes[0].set_xlabel('Количество записей')
for i, v in enumerate(df_sorted['train']):
    axes[0].text(v + 0.3, i, str(v), va='center', fontsize=9)

df_vsorted = df_counts.sort_values('valid', ascending=True)
axes[1].barh(df_vsorted['class'], df_vsorted['valid'], color='#f77f00')
axes[1].set_title('Valid — записей по классу', fontsize=14)
axes[1].set_xlabel('Количество записей')
for i, v in enumerate(df_vsorted['valid']):
    axes[1].text(v + 0.1, i, str(v), va='center', fontsize=9)

plt.tight_layout()
plt.savefig('../training_logs/class_distribution.png', dpi=100, bbox_inches='tight')
plt.show()

## 4. Анализ аудиосигналов

In [ ]:
from ml.preprocess import SAMPLE_RATE

print(f'Sample rate (assumed): {SAMPLE_RATE} Hz')
print(f'Длина сигнала: {train_x.shape[1]} сэмплов')
print(f'Длительность: {train_x.shape[1] / SAMPLE_RATE:.2f} сек')
print(f'Форма одной записи: {train_x[0].shape}')
print(f'\nАмплитудная статистика (train):')
flat = train_x.reshape(train_x.shape[0], -1)
print(f'  min: {flat.min():.4f}')
print(f'  max: {flat.max():.4f}')
print(f'  mean: {flat.mean():.6f}')
print(f'  std: {flat.std():.4f}')

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(16, 12))

classes_to_show = ['Kepler-186f', 'K2-72e', 'Gliese_12_b', '55_Cancri_Bc']
colors = ['#00b4d8', '#f77f00', '#4cc9f0', '#f72585']

for ax, cls, color in zip(axes, classes_to_show, colors):
    idx = next(i for i, l in enumerate(train_labels) if l == cls)
    signal = train_x[idx].squeeze()
    t = np.linspace(0, len(signal) / SAMPLE_RATE, len(signal))
    ax.plot(t[:2000], signal[:2000], color=color, linewidth=0.5, alpha=0.9)
    ax.set_title(f'Класс: {cls}', fontsize=11)
    ax.set_ylabel('Амплитуда')
    ax.set_xlim(0, t[2000])

axes[-1].set_xlabel('Время (сек)')
plt.suptitle('Временные сигналы (первые 2000 сэмплов)', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('../training_logs/waveforms.png', dpi=100, bbox_inches='tight')
plt.show()

## 5. Mel-спектрограммы

In [ ]:
from ml.preprocess import audio_to_mel

n_show = 5
sample_classes = list(label_map.keys())[:n_show]

fig, axes = plt.subplots(1, n_show, figsize=(20, 4))

for ax, cls in zip(axes, sample_classes):
    idx = next(i for i, l in enumerate(train_labels) if l == cls)
    mel = audio_to_mel(train_x[idx]).squeeze().numpy()
    im = ax.imshow(mel, aspect='auto', origin='lower', cmap='viridis')
    ax.set_title(cls, fontsize=9)
    ax.set_xlabel('Время')
    ax.set_ylabel('Mel bins')

plt.suptitle('Mel-спектрограммы по классам', fontsize=13)
plt.tight_layout()
plt.savefig('../training_logs/mel_spectrograms.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
fig, axes = plt.subplots(4, 5, figsize=(22, 14))
axes_flat = axes.flatten()

for ax, (cls, idx_cls) in zip(axes_flat, label_map.items()):
    sample_idx = next(i for i, l in enumerate(train_labels) if l == cls)
    mel = audio_to_mel(train_x[sample_idx]).squeeze().numpy()
    ax.imshow(mel, aspect='auto', origin='lower', cmap='magma')
    ax.set_title(f'{idx_cls}: {cls}', fontsize=8)
    ax.axis('off')

plt.suptitle('Mel-спектрограммы — все 20 классов', fontsize=14)
plt.tight_layout()
plt.savefig('../training_logs/all_classes_mel.png', dpi=100, bbox_inches='tight')
plt.show()

## 6. PCA / t-SNE — кластеризация по классам

In [ ]:
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler

print('Вычисляю Mel-спектрограммы для всего train...')
features = []
for i, sig in enumerate(train_x):
    mel = audio_to_mel(sig).squeeze().numpy()
    features.append(mel.mean(axis=1))
    if (i + 1) % 200 == 0:
        print(f'  {i+1}/{len(train_x)}')

features = np.array(features)
print('Feature matrix shape:', features.shape)

In [ ]:
scaler = StandardScaler()
features_scaled = scaler.fit_transform(features)

pca = PCA(n_components=50, random_state=42)
features_pca = pca.fit_transform(features_scaled)
print(f'PCA: explained variance (top-10): {pca.explained_variance_ratio_[:10].cumsum()[-1]:.3f}')

tsne = TSNE(n_components=2, perplexity=30, random_state=42, n_iter=1000)
features_2d = tsne.fit_transform(features_pca)
print('t-SNE done')

In [ ]:
y_int = encode_labels(train_y, label_map)
palette = sns.color_palette('husl', 20)

fig, axes = plt.subplots(1, 2, figsize=(18, 8))

pca2 = PCA(n_components=2, random_state=42)
feat_pca2 = pca2.fit_transform(features_scaled)

for cls_name, cls_idx in label_map.items():
    mask = y_int == cls_idx
    axes[0].scatter(feat_pca2[mask, 0], feat_pca2[mask, 1],
                    c=[palette[cls_idx]], label=cls_name, alpha=0.7, s=20)
axes[0].set_title('PCA (2D)', fontsize=13)
axes[0].legend(fontsize=6, ncol=2, loc='best')

for cls_name, cls_idx in label_map.items():
    mask = y_int == cls_idx
    axes[1].scatter(features_2d[mask, 0], features_2d[mask, 1],
                    c=[palette[cls_idx]], label=cls_name, alpha=0.7, s=20)
axes[1].set_title('t-SNE (2D)', fontsize=13)
axes[1].legend(fontsize=6, ncol=2, loc='best')

plt.suptitle('Кластеры классов по усреднённым Mel-спектрограммам', fontsize=14)
plt.tight_layout()
plt.savefig('../training_logs/pca_tsne.png', dpi=100, bbox_inches='tight')
plt.show()

## 7. Статистика амплитуд по классам

In [ ]:
stats_by_class = {}
for cls in label_map:
    idxs = [i for i, l in enumerate(train_labels) if l == cls]
    signals = train_x[idxs].squeeze()
    stats_by_class[cls] = {
        'mean_rms': np.sqrt(np.mean(signals ** 2, axis=1)).mean(),
        'std_rms': np.sqrt(np.mean(signals ** 2, axis=1)).std(),
        'peak': np.abs(signals).max(axis=1).mean(),
    }

df_stats = pd.DataFrame(stats_by_class).T.reset_index()
df_stats.columns = ['class', 'mean_rms', 'std_rms', 'peak']
df_stats = df_stats.sort_values('mean_rms', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].barh(df_stats['class'], df_stats['mean_rms'], xerr=df_stats['std_rms'],
             color='#4cc9f0', ecolor='white', capsize=3, alpha=0.9)
axes[0].set_title('Средний RMS по классам', fontsize=12)
axes[0].set_xlabel('RMS амплитуда')

axes[1].barh(df_stats['class'], df_stats['peak'], color='#f72585', alpha=0.9)
axes[1].set_title('Пиковая амплитуда (средняя) по классам', fontsize=12)
axes[1].set_xlabel('Peak амплитуда')

plt.tight_layout()
plt.savefig('../training_logs/amplitude_stats.png', dpi=100, bbox_inches='tight')
plt.show()

## 8. Спектральная плотность мощности (PSD)

In [ ]:
from scipy import signal as scipy_signal

fig, ax = plt.subplots(figsize=(14, 6))
palette = sns.color_palette('husl', 20)

for cls_name, cls_idx in label_map.items():
    idxs = [i for i, l in enumerate(train_labels) if l == cls_name][:3]
    psds = []
    for i in idxs:
        freq, psd = scipy_signal.welch(train_x[i].squeeze(), fs=SAMPLE_RATE, nperseg=1024)
        psds.append(psd)
    mean_psd = np.mean(psds, axis=0)
    ax.semilogy(freq, mean_psd, color=palette[cls_idx], alpha=0.6, linewidth=0.8, label=cls_name)

ax.set_title('Спектральная плотность мощности (PSD) по классам', fontsize=13)
ax.set_xlabel('Частота (Гц)')
ax.set_ylabel('Мощность (log)')
ax.legend(fontsize=7, ncol=2, loc='upper right')
plt.tight_layout()
plt.savefig('../training_logs/psd.png', dpi=100, bbox_inches='tight')
plt.show()

## 9. Выводы

In [ ]:
print('=== ВЫВОДЫ EDA ===')
print()
print(f'1. Датасет: {len(train_x)} train + {len(valid_x)} valid записей, 20 классов (планет)')
print(f'2. Метки повреждены: MD5-хэш + название планеты → восстановлены regex')
print(f'3. Форма сигнала: {train_x.shape[1]} сэмплов, assumed SR={SAMPLE_RATE}Hz → {train_x.shape[1]/SAMPLE_RATE:.1f}с')
print(f'4. Дисбаланс классов: min={min(train_counts.values())}, max={max(train_counts.values())} (умеренный)')
print(f'5. Mel-спектрограмма: (1, 64, 157) — подходит для CNN')
print(f'6. t-SNE показывает частичную кластеризацию → классы различимы')
print()
print('Рекомендуемые гиперпараметры:')
print('  n_mels=64, sample_rate=22050, n_fft=1024, hop_length=512')
print('  4 conv блока, dropout_conv~0.2-0.3, dropout_fc~0.4-0.5')
print('  lr~1e-3, batch_size=32, epochs=50-80, AdamW + ReduceLROnPlateau')
print('  Augmentation: TimemasK + FreqMask + Gaussian noise')